# M-Shots Learning


In this lab, we'll explore zero-shot, one-shot, and few-shot prompting. We'll compare how examples and explicit instructions influence the consistency and format of model responses.


## Setup

Install the dependencies if needed (`pip install -U openai python-dotenv`). Create a `.env` file containing `OPENAI_API_KEY=your_api_key`, or set the environment variable in your shell. Never commit your API key to version control.


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your environment or .env file.")

client = OpenAI()
MODEL = "gpt-4o-mini"


## Few-shot prompting

A *shot* is an example included in a prompt:

- **Zero-shot:** no examples; the model relies on the instructions and its prior training.
- **One-shot:** one example.
- **Few-shot:** multiple examples.

Examples can help a model follow a desired style, label set, or output format. There is no universally correct number of examples: results depend on the task, the examples, and the model. Examples also consume input tokens, so use a small, representative set and evaluate the results.

This notebook uses OpenAI's **Responses API** with `gpt-4o-mini`. The API accepts a list of role-based messages as `input`; the response text is available through `response.output_text`.


In [ ]:
def return_OAIResponse(user_message, context):
    """Send role-based prompt context and a question to the Responses API."""
    messages = [*context, {"role": "user", "content": f"Question: {user_message}"}]
    response = client.responses.create(
        model=MODEL,
        input=messages,
    )
    return response.output_text


## 1. Zero-shot prompting

With no examples, the model may answer correctly but choose its own format. Inspect the output and note which details are present or missing.


In [ ]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer factual questions clearly."}
]

In [ ]:
# let's see the result with zero-shot:
print(return_OAIResponse("Who won the 2010 F1 Drivers' Championship?", context_user))

## 2. One-shot prompting

Now include one example as a user/assistant exchange. The example demonstrates the format we want; it does not guarantee that every answer will be correct.


In [ ]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2000 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Michael Schumacher\nTeam: Ferrari\nPoints: 108"},
]

In [ ]:
# let's see the result with one-shot:
print(return_OAIResponse("Who won the 2011 F1 Drivers' Championship?", context_user))

## 3. Few-shot prompting

Multiple examples show the format across more than one case. Keep examples accurate and representative of the task.


In [ ]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2010 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Sebastian Vettel\nTeam: Red Bull Racing\nPoints: 256"},
    {"role": "user", "content": "Who won the 2009 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Jenson Button\nTeam: Brawn GP\nPoints: 95"},
]
print(return_OAIResponse("Who won the 2006 F1 Drivers' Championship?", context_user))


In [ ]:
# let's see the result with few-shot:
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))

## 4. Examples as messages versus instructions

Role-based examples make the demonstration clearer: each example question is a `user` message and its expected answer is an `assistant` message. The system message states the task. You can compare this with a prompt that explains the format directly. Neither approach is always best; test both on the same questions.


In [ ]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer with exactly these fields, each on its own line: Driver, Team, Points. Use the championship winner's season-end points."}
]
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))


## 5. Classification with few-shot examples

Few-shot prompting can also be useful for other types of tasks, including classification. By providing examples of inputs and their expected labels, we can guide a model toward the desired behavior.

In this example, we'll classify product reviews by sentiment into three categories: **Positive**, **Negative**, or **Neutral**. Notice how the examples illustrate both the task and the expected output format.


In [ ]:
context_user = [
    {"role": "system", "content": "Classify the product review as Positive, Negative, or Neutral. Return only the label."},
    {"role": "user", "content": "It fulfilled its function perfectly, the price is fair, and I would buy it again."},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "It did not work well and I would not buy it again. It is too expensive."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "I do not know what to say; my son uses it, but he does not love it."},
    {"role": "assistant", "content": "Neutral"},
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))


## Exercise

- Choose a task (for example, extracting fields, classifying feedback, or rewriting text).
    - For example: Extract the product name, price, and category from a product description.
- Create zero-shot, one-shot, and few-shot versions of your prompt.
- Test each version on the same set of at least five inputs.
- Compare correctness, consistency, and adherence to the requested format. Include any failures or hallucinations you observe.
- Write a one-page report summarizing your findings and what you learned.


## Exercise

Your task is to extract structured information from product descriptions.

Use the following product description as your starting example:

> The SoundWave Pro wireless headphones feature noise cancellation, Bluetooth 5.3 connectivity, and up to 30 hours of battery life. Available in black, they cost €89.99 and belong to the Electronics category.

1. Create zero-shot, one-shot, and few-shot prompts to extract the following fields:
   - **Product name**
   - **Price**
   - **Category**

2. Generate three additional product descriptions (using code or an LLM) and test all three prompts on the same descriptions.

3. Compare the results in terms of accuracy, consistency, and adherence to the expected output format. Include any errors or hallucinations you observe.

4. Write a brief, simple report summarizing your findings and what you learned about the impact of examples on model performance.

## Notes

- Model output can vary and may contain factual errors; verify factual answers independently.
- Remember to keep API keys private. API usage may incur charges.


## Exercise solution: product information extraction

The test set below contains the supplied SoundWave Pro description, three additional complete product descriptions, and one missing-information stress test so the prompts are evaluated on the same five inputs. Zero-shot, one-shot, and few-shot use the same task and output contract; they differ only in the number of examples. The evaluation checks exact field matches and whether every answer follows the requested three-line format.

**Running the evaluation makes 15 Responses API requests (five inputs for each of three prompt styles) and may incur API charges.** Run the code cell only when your API key is configured and you are ready to make those requests. The report is generated from the actual responses, so it does not assume or fabricate model results.

In [ ]:
import json
import re

test_cases = [
    {
        "description": "The SoundWave Pro wireless headphones feature noise cancellation, Bluetooth 5.3 connectivity, and up to 30 hours of battery life. Available in black, they cost €89.99 and belong to the Electronics category.",
        "expected": {"Product name": "SoundWave Pro", "Price": "€89.99", "Category": "Electronics"},
    },
    {
        "description": "The BrewBuddy Mini espresso maker has a compact stainless-steel design and a removable water tank. It costs €129.50 and is in the Kitchen Appliances category.",
        "expected": {"Product name": "BrewBuddy Mini", "Price": "€129.50", "Category": "Kitchen Appliances"},
    },
    {
        "description": "The TrailLite 20L hiking backpack includes a rain cover and padded shoulder straps. The price is $54.00, and its category is Outdoor Gear.",
        "expected": {"Product name": "TrailLite 20L", "Price": "$54.00", "Category": "Outdoor Gear"},
    },
    {
        "description": "The PaperNest A5 Dotted Notebook has dotted pages and a durable cover. This stationery item costs £12.99.",
        "expected": {"Product name": "PaperNest A5 Dotted Notebook", "Price": "£12.99", "Category": "Stationery"},
    },
    {
        "description": "The GlowDesk LED Lamp provides adjustable brightness and a flexible neck. The description does not state its price or product category.",
        "expected": {"Product name": "GlowDesk LED Lamp", "Price": "Not specified", "Category": "Not specified"},
    },
]

system_instruction = (
    "Extract the product name, price, and category from the product description. "
    "Return exactly three lines, one field per line, with no extra text. Use these labels and this order: "
    "Product name: <value>, Price: <value>, Category: <value>. "
    "Copy values from the description. If a field is absent, write Not specified; do not guess."
)
base_context = [{"role": "system", "content": system_instruction}]
one_shot_context = [
    *base_context,
    {"role": "user", "content": "The CloudSip Steel Bottle keeps drinks cold for 24 hours. It costs $24.50 and belongs to Home & Kitchen."},
    {"role": "assistant", "content": "Product name: CloudSip Steel Bottle\nPrice: $24.50\nCategory: Home & Kitchen"},
]
few_shot_context = [
    *one_shot_context,
    {"role": "user", "content": "The MapleDesk Organizer is made from bamboo and costs €18.00 in the Office Supplies category."},
    {"role": "assistant", "content": "Product name: MapleDesk Organizer\nPrice: €18.00\nCategory: Office Supplies"},
    {"role": "user", "content": "The Luna Reading Light is rechargeable and listed in Lighting. Its price is not provided."},
    {"role": "assistant", "content": "Product name: Luna Reading Light\nPrice: Not specified\nCategory: Lighting"},
]
prompt_styles = {
    "zero-shot": base_context,
    "one-shot": one_shot_context,
    "few-shot": few_shot_context,
}

field_order = ["Product name", "Price", "Category"]
field_pattern = re.compile(r"^(Product name|Price|Category):\s*(.*?)\s*$", re.MULTILINE)

def normalize(value):
    return " ".join(value.casefold().split())

def evaluate_format_and_fields(response_text, expected):
    lines = response_text.strip().splitlines()
    matches = field_pattern.findall(response_text)
    format_ok = len(lines) == 3 and [field for field, _ in matches] == field_order
    extracted = dict(matches)
    correct_fields = {
        field: field in extracted and normalize(extracted[field]) == normalize(expected[field])
        for field in field_order
    }
    return extracted, format_ok, correct_fields

evaluation_results = []
for style, context in prompt_styles.items():
    for case_number, case in enumerate(test_cases, start=1):
        response_text = return_OAIResponse(case["description"], context)
        extracted, format_ok, correct_fields = evaluate_format_and_fields(response_text, case["expected"])
        evaluation_results.append({
            "prompt_style": style,
            "case": case_number,
            "expected": case["expected"],
            "response": response_text,
            "extracted": extracted,
            "format_ok": format_ok,
            "correct_fields": correct_fields,
        })

summary = {}
for style in prompt_styles:
    rows = [row for row in evaluation_results if row["prompt_style"] == style]
    total_fields = len(test_cases) * len(field_order)
    correct_count = sum(sum(row["correct_fields"].values()) for row in rows)
    summary[style] = {
        "field_accuracy": correct_count / total_fields,
        "exact_record_accuracy": sum(all(row["correct_fields"].values()) for row in rows) / len(test_cases),
        "format_adherence": sum(row["format_ok"] for row in rows) / len(test_cases),
        "correct_fields": correct_count,
        "total_fields": total_fields,
    }

best_accuracy = max(metrics["field_accuracy"] for metrics in summary.values())
top_styles = [style for style, metrics in summary.items() if metrics["field_accuracy"] == best_accuracy]
metric_lines = [
    f"- **{style}:** field accuracy {metrics['correct_fields']}/{metrics['total_fields']} "
    f"({metrics['field_accuracy']:.0%}); exact records {metrics['exact_record_accuracy']:.0%}; "
    f"format adherence {metrics['format_adherence']:.0%}."
    for style, metrics in summary.items()
]
failures = [
    f"- {row['prompt_style']} / input {row['case']} / {field}: expected "
    f"{row['expected'][field]!r}, got {row['extracted'].get(field, '<missing>')!r}"
    for row in evaluation_results
    for field, is_correct in row["correct_fields"].items()
    if not is_correct
]
hallucinations = [
    f"- {row['prompt_style']} / input {row['case']} / {field}: "
    f"expected 'Not specified', got {row['extracted'].get(field, '<missing>')!r}"
    for row in evaluation_results
    for field in ("Price", "Category")
    if row["expected"][field] == "Not specified"
    and normalize(row["extracted"].get(field, "<missing>")) != "not specified"
]

report = "\n".join([
    "# Results and brief report",
    "",
    "The same five product descriptions were tested with zero-shot, one-shot, and few-shot prompts. "
    "The one-shot prompt included one labeled example; the few-shot prompt included three, including an absent-value example. "
    "Each style was evaluated on 15 expected fields. Field accuracy counts correct values, exact-record accuracy "
    "requires all three fields to be correct for an input, and format adherence requires exactly three correctly ordered lines.",
    "",
    *metric_lines,
    "",
    f"The highest field accuracy was achieved by: {', '.join(top_styles)}. "
    "Failures below identify incorrect or missing extractions; a wrong value for an unstated field is a hallucination.",
    "",
    "## Incorrect fields",
    *(failures or ["- None." ]),
    "",
    "## Hallucinations on unstated values",
    *(hallucinations or ["- None observed." ]),
    "",
    "## What I learned",
    "Examples can demonstrate the expected labels and format, and the few-shot absent-value example may help discourage guessing. "
    "However, examples do not guarantee correctness: compare the measured accuracy and format adherence rather than assuming more examples are always better. "
    "This small test uses one response per prompt and input, so format adherence is only a consistency proxy; repeated trials would be needed "
    "to measure response-to-response consistency or make broad claims about model performance.",
])

print(json.dumps(evaluation_results, ensure_ascii=False, indent=2))
print(report)
